# Financial Metrics - Latest WSB Stocks
This notebook refreshes financial metrics for the latest WallStreetBets tickers using current Yahoo Finance data.
It now:
- loads the latest top WSB stocks,
- fetches current Yahoo Finance price history,
- computes rolling standard deviation, SMA, EWMA, RSI, and MACD,
- runs a linear regression / RMSE check for each selected ticker,
- saves refreshed CSVs and charts under `outputs/latest_wsb_analysis/`.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    compute_price_features,
    evaluate_linear_regression,
    fetch_price_history,
    plot_symbol_dashboard,
    run_pipeline,
)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print(f"Project root: {ROOT}")

Project root: /home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
CHARTS_DIR = OUTPUT_DIR / "charts"
PRICES_DIR = OUTPUT_DIR / "prices"
PRICE_PERIOD = "1y"
TOP_K = 4
import os
REFRESH_WSB_DATA = os.getenv("WSB_REFRESH_DATA", "0") != "0"
MANUAL_TICKERS = None  # Example: ["AMD", "TSLA"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
PRICES_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Refresh or load the latest WSB ranking

In [3]:
summary_path = OUTPUT_DIR / "top10_wsb_stocks.csv"
if REFRESH_WSB_DATA or (not summary_path.exists()):
    pipeline_result = run_pipeline(top_n=10, per_feed=100, price_period=PRICE_PERIOD, output_dir=OUTPUT_DIR)
    print(json.dumps(pipeline_result, indent=2))
summary_df = cast(pd.DataFrame, pd.read_csv(str(summary_path)))
summary_df["latest_mention"] = pd.to_datetime(summary_df["latest_mention"], utc=True)
summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "latest_mention"]]

,ticker,mention_count,post_count,avg_sentiment,latest_mention
0,ASTS,79,7,0.379900,2026-05-27 20:46:36+00:00
1,NVDA,21,11,0.121309,2026-05-27 20:31:16+00:00
2,RKLB,15,9,0.325089,2026-05-27 23:14:45+00:00
3,AMD,9,9,0.300078,2026-05-27 02:32:08+00:00
4,LUNR,18,6,0.616450,2026-05-26 23:17:49+00:00
5,MSFT,9,8,0.372287,2026-05-27 21:52:23+00:00
6,MU,10,6,0.597800,2026-05-27 19:36:23+00:00
7,SNDK,8,6,0.654300,2026-05-27 19:36:23+00:00
8,CRSR,8,5,0.522040,2026-05-27 20:55:23+00:00
9,TSLA,7,5,0.278180,2026-05-27 03:02:32+00:00


## Step 4 - Select tickers for financial analysis

In [4]:
if MANUAL_TICKERS:
    selected_tickers = [ticker.upper() for ticker in MANUAL_TICKERS]
else:
    selected_tickers = summary_df["ticker"].head(TOP_K).tolist()
print("Selected tickers:", selected_tickers)

Selected tickers: ['ASTS', 'NVDA', 'RKLB', 'AMD']


## Step 5 - Fetch current price history and compute financial metrics

In [5]:
price_feature_frames = {}
regression_frames = {}
analysis_rows = []
for ticker in selected_tickers:
    history_df = fetch_price_history(ticker, PRICE_PERIOD)
    features_df = compute_price_features(history_df)
    rmse, regression_df = evaluate_linear_regression(features_df)
    features_path = PRICES_DIR / f"{ticker.lower()}_price_features.csv"
    regression_path = PRICES_DIR / f"{ticker.lower()}_regression_results.csv"
    dashboard_path = CHARTS_DIR / f"{ticker.lower()}_dashboard.png"
    features_df.to_csv(features_path)
    regression_df.to_csv(regression_path)
    plot_symbol_dashboard(ticker, features_df, regression_df, dashboard_path)
    price_feature_frames[ticker] = features_df
    regression_frames[ticker] = regression_df
    analysis_rows.append({
        "ticker": ticker,
        "rows": int(len(features_df)),
        "last_close": float(features_df["Close"].dropna().iloc[-1]),
        "latest_rsi": float(features_df["RSI"].dropna().iloc[-1]),
        "latest_macd": float(features_df["MACD"].dropna().iloc[-1]),
        "return_21d": float(features_df["21D Return"].dropna().iloc[-1]),
        "rmse": rmse,
        "features_csv": str(features_path),
        "regression_csv": str(regression_path),
        "dashboard_png": str(dashboard_path),
    })
analysis_df = pd.DataFrame(analysis_rows).sort_values("ticker").reset_index(drop=True)
analysis_summary_path = OUTPUT_DIR / "financial_metrics_summary.csv"
analysis_df.to_csv(analysis_summary_path, index=False)
analysis_df

,ticker,rows,last_close,latest_rsi,latest_macd,return_21d,rmse,features_csv,regression_csv,dashboard_png
0,AMD,251,495.540009,74.078809,47.007379,0.480859,16.535870,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
1,ASTS,251,129.600006,76.966748,8.483142,0.678757,5.355276,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
2,NVDA,251,212.600006,51.039990,5.214426,-0.018513,2.732572,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...
3,RKLB,251,150.229996,73.570757,16.425138,0.825617,5.446622,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...,/home/runner/work/Project2-Sentiment-analysis-...


## Step 6 - Inspect the computed metrics for each selected ticker

In [6]:
for ticker in selected_tickers:
    print(f"\n===== {ticker} latest metrics =====")
    display_cols = [
        "Close",
        "Volume",
        "30-Day Rolling STD",
        "30-Day Rolling SMA",
        "30-Day Rolling EWMA",
        "RSI",
        "MACD",
        "5D Return",
        "21D Return",
    ]
    print(price_feature_frames[ticker][display_cols].tail(5).to_string())


===== ASTS latest metrics =====
                 Close    Volume  30-Day Rolling STD  30-Day Rolling SMA  30-Day Rolling EWMA        RSI      MACD  5D Return  21D Return
Date                                                                                                                                     
2026-05-20   89.580002  20010000            9.002649           80.228667            83.085541  58.107507  0.839943   0.197434    0.119610
2026-05-21   96.230003  24819900            9.240428           80.382667            83.386739  62.590474  1.964602   0.159258    0.136664
2026-05-22  105.860001  30710500           10.018117           80.748000            83.901663  67.940704  3.591561   0.265209    0.344254
2026-05-26  119.699997  48352900           11.863316           81.439000            84.721841  73.750991  5.929360   0.378556    0.566754
2026-05-27  129.600006  34852084           14.732003           82.806667            85.749974  76.966748  8.483142   0.471056    0.678757



## Step 7 - Plot the latest closing prices for the selected tickers

In [7]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in selected_tickers:
    close_series = price_feature_frames[ticker]["Close"].dropna()
    ax.plot(close_series.index, close_series.values, label=ticker, linewidth=1.8)
ax.set_title("Latest selected WSB tickers - close prices")
ax.set_ylabel("Close")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8 - Review the regression predictions

In [8]:
for ticker in selected_tickers:
    regression_df = regression_frames[ticker]
    print(f"\n===== {ticker} regression results =====")
    if regression_df.empty:
        print("Not enough data for regression output.")
    else:
        print(regression_df.tail(10).to_string())


===== ASTS regression results =====
                 Close  Predicted Close
Date                                   
2026-05-13   74.809998        81.123852
2026-05-14   83.010002        89.036993
2026-05-15   83.669998        86.977462
2026-05-18   86.830002        88.047344
2026-05-19   88.099998        93.576651
2026-05-20   89.580002        95.152282
2026-05-21   96.230003        97.484309
2026-05-22  105.860001       105.139443
2026-05-26  119.699997       115.025386
2026-05-27  129.600006       124.157984

===== NVDA regression results =====
                 Close  Predicted Close
Date                                   
2026-05-13  225.830002       222.134753
2026-05-14  235.740005       228.863796
2026-05-15  225.320007       221.990341
2026-05-18  222.320007       220.760347
2026-05-19  220.610001       220.799154
2026-05-20  223.470001       223.562691
2026-05-21  219.509995       220.636500
2026-05-22  215.330002       219.237482
2026-05-26  214.860001       218.777689
2026-0

## Step 9 - Final notebook summary

In [9]:
notebook_summary = {
    "selected_tickers": selected_tickers,
    "price_period": PRICE_PERIOD,
    "top_k": TOP_K,
    "analysis_summary_csv": str(analysis_summary_path.resolve()),
    "output_dir": str(OUTPUT_DIR.resolve()),
}
print(json.dumps(notebook_summary, indent=2))

{
  "selected_tickers": [
    "ASTS",
    "NVDA",
    "RKLB",
    "AMD"
  ],
  "price_period": "1y",
  "top_k": 4,
  "analysis_summary_csv": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis/financial_metrics_summary.csv",
  "output_dir": "/home/runner/work/Project2-Sentiment-analysis-WSbets/Project2-Sentiment-analysis-WSbets/outputs/latest_wsb_analysis"
}
